# Bradford Bulls — High-Value Frame Selector

Mục tiêu: từ mỗi video trận đấu (`M<id>_<target_color>_<res>.mp4`), tự động trích ra **~200 frame chất lượng cao** trên cầu thủ của **đội target** để mang đi annotate logo trên Roboflow.

## Pipeline 8 stage
1. **Sample** 1 frame/giây bằng `cv2.VideoCapture` (không decode toàn bộ).
2. **Auto static-overlay mask** per-video: median + std của ~300 frame → mask scoreboard, BullsTV watermark, v.v.
3. **Shot-type gating**: grass ratio (HSV green) loại jumbotron/replay/khán đài đặc.
4. **Person detection**: YOLOv8n trên GPU, giữ bbox đủ lớn (≥ 8% frame height).
5. **Target-color filter**: HSV match jersey ROI từng bbox → đếm số target player.
6. **Weighted Laplacian sharpness**: chỉ trên jersey vùng target-player; mask overlay & bbox đối thủ; weight 3× torso.
7. **Dedup**: temporal NMS + pHash (imagehash).
8. **Export** top-K JPG ở độ phân giải gốc + manifest CSV + sidecar JSON (bbox hint).

Chạy trên Google Colab với T4/A100 GPU. Toàn bộ I/O qua Google Drive.

## 1. Cài đặt & mount Drive

In [ ]:
!pip -q install ultralytics==8.3.40 imagehash==4.3.1 tqdm pandas opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 2. Configuration

Đặt videos vào `MyDrive/bradford_bulls/videos/`. Convention tên: `M<id>_<color>_<res>.mp4` — color sẽ được tự động parse.

**Supported colors:** `white`, `black`, `red`, `blue`, `yellow`, `claret`, `green`, `orange`. Nếu bạn muốn thêm preset khác, sửa `COLOR_PRESETS` ở cell tiếp theo.

In [ ]:
from pathlib import Path

# === Đường dẫn ===
ROOT       = Path('/content/drive/MyDrive/bradford_bulls')
VIDEO_DIR  = ROOT / 'videos'
OUT_DIR    = ROOT / 'extracted_frames'
MASK_DIR   = ROOT / 'overlay_masks'
CACHE_DIR  = ROOT / 'cache_candidates'
for d in [OUT_DIR, MASK_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# === Sampling & target K ===
# SAMPLE_FPS_OVERRIDE=None -> adaptive theo duration (xem cell 'process_video'):
#   < 20min (highlight)  : 2.0 fps
#   20-60min             : 1.0 fps
#   1-2h                 : 0.5 fps
#   2h+ (full match)     : 0.33 fps
SAMPLE_FPS_OVERRIDE = None
TARGET_K            = 250    # số frame ưu tiên xuất ra per video
MIN_OUTPUT          = 150    # YIELD GUARANTEE — nếu chọn được < số này tự nới dedup theo 3 tier

# === Overlay-mask builder ===
MASK_N_SAMPLES = 300    # số frame dùng để học static overlay
MASK_STD_THR   = 8.0    # std threshold (0-255 grayscale) — pixel std thấp hơn coi là static

# === Person detection / target filter ===
YOLO_WEIGHTS       = 'yolov8n.pt'
YOLO_CONF          = 0.35
MIN_BBOX_H         = 0.08    # bbox height >= 8% frame height để được xét
MIN_PERSONS        = 2       # tối thiểu 2 person bbox đủ lớn (filter chung)
MIN_TARGET_PLAYERS = 1       # giảm xuống 1: close-up 1 cầu thủ vẫn cực giá trị cho logo annotation
MAX_TARGET_H_REQ   = 0.18    # HARD GATE — ít nhất 1 target bbox phải >= 18% frame height (loại wide shot)
JERSEY_MATCH_THR   = 0.28    # tỉ lệ pixel jersey ROI khớp màu (slightly relaxed)
CLOSE_UP_H         = 0.40    # bbox h >= 40% frame -> đây là close-up, được bonus điểm

# === Sharpness & composite weights ===
W_SHARP    = 0.7
W_NTARGET  = 1.0
W_TGT_H    = 8.0     # tăng 5 -> 8: ưu tiên frame có player to (medium/close)
W_CROWD    = 2.0
W_CLOSEUP  = 3.0     # bonus mạnh cho close-up

# === Dedup ===
PHASH_SIZE        = 16
PHASH_HAM_THR     = 10    # Hamming distance min giữa 2 frame đã chọn
TEMPORAL_WINDOW_S = 2.0   # trong cửa sổ 2s không lấy quá 1 frame

# === Debug ===
MAX_FRAMES_DEBUG = None   # đặt số nhỏ (e.g. 300) để test nhanh, None = full video

print('Configured. Video dir:', VIDEO_DIR)
print('Videos found:')
for v in sorted(VIDEO_DIR.glob('*.mp4')):
    print(' -', v.name)

## 3. Color presets & helper functions

In [ ]:
import re, cv2, json
import numpy as np

# HSV ranges (OpenCV: H 0-179, S 0-255, V 0-255). Tested for broadcast footage.
COLOR_PRESETS = {
    'white':  [((0,   0, 180), (179,  55, 255))],
    'black':  [((0,   0,   0), (179, 110,  65))],
    'red':    [((0,  90,  60), ( 10, 255, 255)),
               ((168, 90,  60), (179, 255, 255))],
    'blue':   [((95,  80,  50), (130, 255, 255))],
    'yellow': [((18,  90, 100), ( 35, 255, 255))],
    'orange': [((8,  120, 120), ( 20, 255, 255))],
    'claret': [((160, 80,  40), (179, 255, 170)),
               ((0,   80,  40), (  8, 255, 170))],
    'green':  [((35, 100,  40), ( 85, 255, 200))],
}

GRASS_HSV_LOW  = np.array([30,  35, 25])
GRASS_HSV_HIGH = np.array([90, 255, 230])

def parse_target_color(filename: str) -> str:
    """Parse `M<id>_<color>_<res>.mp4`. Fallback: scan for known color name."""
    name = Path(filename).stem.lower()
    m = re.match(r'^m\d+_([a-z]+)_', name)
    if m and m.group(1) in COLOR_PRESETS:
        return m.group(1)
    for c in COLOR_PRESETS:
        if c in name:
            return c
    raise ValueError(f'Cannot parse target color from {filename}. Rename to M01_white_1080p.mp4 style.')

def color_mask_hsv(hsv: np.ndarray, color: str) -> np.ndarray:
    """Return uint8 0/255 mask of pixels matching `color` in an HSV image."""
    out = None
    for (lo, hi) in COLOR_PRESETS[color]:
        m = cv2.inRange(hsv, np.array(lo, dtype=np.uint8), np.array(hi, dtype=np.uint8))
        out = m if out is None else cv2.bitwise_or(out, m)
    return out

def grass_mask(hsv: np.ndarray) -> np.ndarray:
    return cv2.inRange(hsv, GRASS_HSV_LOW, GRASS_HSV_HIGH)

# Quick sanity check
for f in sorted(VIDEO_DIR.glob('*.mp4')):
    print(f'{f.name:35s} -> target = {parse_target_color(f.name)}')

## 4. Auto static-overlay mask (per video)

Lấy std pixel theo thời gian. Vùng std thấp = scoreboard, watermark, logo overlay đứng yên. Mask này dùng để (a) loại khỏi tính sharpness, (b) loại khỏi grass detection để tránh nhiễu.

In [ ]:
from tqdm.auto import tqdm

def build_overlay_mask(video_path: Path,
                      n_samples: int = MASK_N_SAMPLES,
                      std_thr: float = MASK_STD_THR) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise RuntimeError(f'Cannot read frame count from {video_path}')
    indices = np.linspace(int(total*0.05), int(total*0.95), n_samples).astype(int)
    samples = []
    for idx in tqdm(indices, desc=f'mask {video_path.name}', leave=False):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, f = cap.read()
        if not ok: continue
        samples.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32))
    cap.release()
    if len(samples) < 30:
        raise RuntimeError('Too few samples to build overlay mask')
    stack = np.stack(samples, axis=0)
    std_map = stack.std(axis=0)
    mask = (std_map < std_thr).astype(np.uint8) * 255
    # close small holes; opening removes single-pixel noise
    k1 = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
    k2 = cv2.getStructuringElement(cv2.MORPH_RECT, (15,15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k2)
    # keep only large connected components (≥ 0.3% of frame)
    n, lbl, stats, _ = cv2.connectedComponentsWithStats(mask)
    keep = np.zeros_like(mask)
    min_area = int(0.003 * mask.size)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            keep[lbl == i] = 255
    return keep

# Build masks for all videos and save preview
MASKS = {}
for vpath in sorted(VIDEO_DIR.glob('*.mp4')):
    mask_file = MASK_DIR / f'{vpath.stem}_mask.png'
    if mask_file.exists():
        print(f'Mask cached: {mask_file.name}')
        MASKS[vpath.name] = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)
    else:
        m = build_overlay_mask(vpath)
        cv2.imwrite(str(mask_file), m)
        MASKS[vpath.name] = m
        print(f'Saved mask -> {mask_file.name}  ({(m>0).mean()*100:.1f}% of frame is static overlay)')

In [ ]:
# Optional: visualize overlay masks superimposed on a sample frame
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(MASKS), 2, figsize=(14, 4*len(MASKS)))
if len(MASKS) == 1: axes = axes.reshape(1, -1)
for i, (vname, mask) in enumerate(MASKS.items()):
    cap = cv2.VideoCapture(str(VIDEO_DIR / vname))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)*0.5))
    ok, f = cap.read(); cap.release()
    if not ok: continue
    rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
    overlay = rgb.copy()
    overlay[mask > 0] = (255, 0, 0)
    blended = cv2.addWeighted(rgb, 0.55, overlay, 0.45, 0)
    axes[i,0].imshow(rgb); axes[i,0].set_title(f'{vname} — mid-frame'); axes[i,0].axis('off')
    axes[i,1].imshow(blended); axes[i,1].set_title('Static overlay (red)'); axes[i,1].axis('off')
plt.tight_layout(); plt.show()

## 5. YOLOv8n person detector

In [ ]:
import torch
from ultralytics import YOLO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

yolo = YOLO(YOLO_WEIGHTS)
yolo.to(DEVICE)
_ = yolo.predict(np.zeros((640,640,3), dtype=np.uint8), verbose=False, classes=[0])  # warmup

## 6. Per-frame scoring

Trả `None` cho frame bị loại; trả dict scores cho frame hợp lệ. Stores target/other bbox để dùng làm Roboflow hint.

In [ ]:
def score_frame(frame_bgr: np.ndarray,
                overlay_mask: np.ndarray,
                target_color: str):
    H, W = frame_bgr.shape[:2]
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)

    # --- Stage 3: grass ratio gating ---
    gmask = grass_mask(hsv)
    gmask[overlay_mask > 0] = 0
    grass_ratio = float((gmask > 0).mean())
    if grass_ratio < 0.18:
        return None  # jumbotron, replay, close-up trên 1 mặt, fade, etc.
    if grass_ratio > 0.92:
        return None  # empty pitch / aerial pre-kick

    not_grass = (gmask == 0)
    not_overlay = (overlay_mask == 0)
    upper = (not_grass & not_overlay)[:H//2, :]
    crowd_score = float(upper.mean())
    if crowd_score > 0.85:
        return None  # mostly stands

    # --- Stage 4: persons ---
    res = yolo.predict(frame_bgr, classes=[0], verbose=False,
                       imgsz=640, conf=YOLO_CONF, device=DEVICE)
    boxes = res[0].boxes
    if boxes is None or len(boxes) == 0:
        return None
    xyxy = boxes.xyxy.cpu().numpy()
    heights = xyxy[:,3] - xyxy[:,1]
    big = heights >= H * MIN_BBOX_H
    if big.sum() < MIN_PERSONS:
        return None
    big_boxes = xyxy[big].astype(int)

    # --- Stage 5: target-color classification per bbox ---
    target_boxes, other_boxes = [], []
    for (x1,y1,x2,y2) in big_boxes:
        x1 = max(0, x1); y1 = max(0, y1)
        x2 = min(W, x2); y2 = min(H, y2)
        bh, bw = y2-y1, x2-x1
        if bh < 20 or bw < 8:
            continue
        jy1 = y1 + int(0.20*bh); jy2 = y1 + int(0.55*bh)
        jx1 = x1 + int(0.15*bw); jx2 = x2 - int(0.15*bw)
        if jy2 <= jy1 or jx2 <= jx1:
            other_boxes.append((x1,y1,x2,y2)); continue
        roi_hsv = hsv[jy1:jy2, jx1:jx2]
        roi_overlay = overlay_mask[jy1:jy2, jx1:jx2]
        m = color_mask_hsv(roi_hsv, target_color)
        m[roi_overlay > 0] = 0
        ratio = (m > 0).sum() / max(1, m.size)
        if ratio >= JERSEY_MATCH_THR:
            target_boxes.append((x1,y1,x2,y2))
        else:
            other_boxes.append((x1,y1,x2,y2))

    if len(target_boxes) < MIN_TARGET_PLAYERS:
        return None

    # --- HARD GATE: phải có ít nhất 1 target player to (loại wide shot) ---
    max_target_h = max((b[3]-b[1])/H for b in target_boxes)
    if max_target_h < MAX_TARGET_H_REQ:
        return None

    # --- Stage 6: weighted Laplacian sharpness ---
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_sq = lap * lap

    weight = np.zeros((H, W), dtype=np.float32)
    weight[gmask > 0] = 0.1
    for (x1,y1,x2,y2) in target_boxes:
        bh = y2-y1
        ty1 = y1 + int(0.15*bh); ty2 = y1 + int(0.70*bh)
        weight[ty1:ty2, x1:x2] = 3.0
    for (x1,y1,x2,y2) in other_boxes:
        weight[y1:y2, x1:x2] = 0.0
    weight[overlay_mask > 0] = 0.0
    if weight.sum() < 200:
        return None
    sharp_w = float((lap_sq * weight).sum() / weight.sum())

    avg_target_h = float(np.mean([(b[3]-b[1])/H for b in target_boxes]))
    closeup_bonus = W_CLOSEUP if max_target_h >= CLOSE_UP_H else 0.0

    composite = (
        W_SHARP   * np.log1p(sharp_w) +
        W_NTARGET * len(target_boxes) +
        W_TGT_H   * avg_target_h +
        closeup_bonus -
        W_CROWD   * crowd_score
    )

    return dict(
        sharp_weighted = sharp_w,
        n_target       = len(target_boxes),
        n_other        = len(other_boxes),
        avg_target_h   = avg_target_h,
        max_target_h   = float(max_target_h),
        is_closeup     = bool(max_target_h >= CLOSE_UP_H),
        crowd_score    = crowd_score,
        grass_ratio    = grass_ratio,
        composite      = float(composite),
        target_boxes   = [list(map(int, b)) for b in target_boxes],
        other_boxes    = [list(map(int, b)) for b in other_boxes],
    )

## 7. Process all videos → candidate cache

Mỗi video lưu candidates (frame_idx, scores, pHash, bbox) vào JSON cache để có thể tinh chỉnh top-K mà không cần chạy lại detection.

In [ ]:
import imagehash
from PIL import Image

def phash_int(frame_bgr: np.ndarray, size: int = PHASH_SIZE) -> str:
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return str(imagehash.phash(Image.fromarray(rgb), hash_size=size))

def adaptive_sample_fps(duration_s: float) -> float:
    """Highlight ngắn cần sample dày, full match 2-3h sample thưa để chạy nhanh."""
    if duration_s < 1200:    return 2.0   # < 20min — highlight
    elif duration_s < 3600:  return 1.0   # 20-60min
    elif duration_s < 7200:  return 0.5   # 1-2h
    else:                    return 0.33  # 2h+ full match

def process_video(vpath: Path):
    target_color = parse_target_color(vpath.name)
    overlay = MASKS[vpath.name]
    cap = cv2.VideoCapture(str(vpath))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_s = total / fps
    target_fps = SAMPLE_FPS_OVERRIDE if SAMPLE_FPS_OVERRIDE else adaptive_sample_fps(duration_s)
    step = max(1, int(round(fps / target_fps)))
    indices = list(range(0, total, step))
    if MAX_FRAMES_DEBUG: indices = indices[:MAX_FRAMES_DEBUG]

    cache_file = CACHE_DIR / f'{vpath.stem}_candidates.json'
    if cache_file.exists():
        print(f'[{vpath.name}] candidates cached: {cache_file.name}')
        return json.loads(cache_file.read_text())

    print(f'[{vpath.name}] target={target_color}  duration={duration_s/60:.1f}min  '
          f'fps={fps:.2f}  sample_fps={target_fps:.2f}  step={step}  samples={len(indices)}')
    candidates = []
    for idx in tqdm(indices, desc=vpath.name):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok: continue
        scores = score_frame(frame, overlay, target_color)
        if scores is None: continue
        scores['frame_idx']  = int(idx)
        scores['time_s']     = float(idx) / fps
        scores['phash']      = phash_int(frame)
        candidates.append(scores)
    cap.release()
    cache_file.write_text(json.dumps(candidates))
    n_closeup = sum(1 for c in candidates if c.get('is_closeup'))
    print(f'  -> kept {len(candidates)} candidates ({len(candidates)/max(1,len(indices))*100:.1f}%), '
          f'of which {n_closeup} close-ups')
    return candidates

ALL_CANDIDATES = {}
for vpath in sorted(VIDEO_DIR.glob('*.mp4')):
    ALL_CANDIDATES[vpath.name] = process_video(vpath)

## 8. Dedup + select top-K per video → export

Greedy: duyệt theo composite score giảm dần, chấp nhận nếu (a) không nằm trong cửa sổ thời gian 2s với frame đã chọn, (b) pHash Hamming ≥ 10 so với mọi frame đã chọn.

In [ ]:
import pandas as pd

def hamming(h1: str, h2: str) -> int:
    return imagehash.hex_to_hash(h1) - imagehash.hex_to_hash(h2)

def select_topk(candidates, k: int,
                ham_thr: int = PHASH_HAM_THR,
                temporal_s: float = TEMPORAL_WINDOW_S):
    cands = sorted(candidates, key=lambda c: c['composite'], reverse=True)
    chosen, chosen_hashes, chosen_times = [], [], []
    for c in cands:
        if any(abs(c['time_s'] - t) < temporal_s for t in chosen_times):
            continue
        if any(hamming(c['phash'], h) < ham_thr for h in chosen_hashes):
            continue
        chosen.append(c)
        chosen_hashes.append(c['phash'])
        chosen_times.append(c['time_s'])
        if len(chosen) >= k:
            break
    return sorted(chosen, key=lambda c: c['frame_idx'])

def select_with_yield_guarantee(candidates, k: int, min_out: int):
    """3-tier relaxation: trả về ít nhất min_out frame nếu candidates đủ.
    Mỗi tier nới temporal/pHash threshold thêm khi pass trước < min_out."""
    tiers = [
        ('strict',         PHASH_HAM_THR, TEMPORAL_WINDOW_S),
        ('relax_temporal', PHASH_HAM_THR, 1.0),
        ('relax_dedup',    6,             1.0),
        ('minimal_dedup',  3,             0.5),
    ]
    for name, ham, tw in tiers:
        sel = select_topk(candidates, k, ham_thr=ham, temporal_s=tw)
        if len(sel) >= min_out:
            return sel, name
    return sel, 'exhausted'  # trả pass cuối kể cả < min_out

def export_frames(vpath: Path, selected):
    out = OUT_DIR / vpath.stem
    out.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(vpath))
    rows = []
    for c in tqdm(selected, desc=f'export {vpath.stem}'):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(c['frame_idx']))
        ok, frame = cap.read()
        if not ok: continue
        ts = c['time_s']; mm, ss = divmod(int(ts), 60); hh, mm = divmod(mm, 60)
        stamp = f'{hh:02d}{mm:02d}{ss:02d}_{int((ts%1)*1000):03d}'
        fname = f'{vpath.stem}_f{c["frame_idx"]:08d}_{stamp}.jpg'
        cv2.imwrite(str(out / fname), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
        keep_keys = ['frame_idx','time_s','composite','sharp_weighted',
                    'n_target','n_other','avg_target_h','max_target_h',
                    'is_closeup','crowd_score','grass_ratio',
                    'target_boxes','other_boxes']
        sidecar = {k: c[k] for k in keep_keys if k in c}
        sidecar['source_video'] = vpath.name
        (out / fname.replace('.jpg', '.json')).write_text(json.dumps(sidecar, indent=2))
        rows.append(dict(filename=fname, **{k: c.get(k) for k in
            ['frame_idx','time_s','composite','sharp_weighted','n_target','max_target_h','is_closeup','crowd_score']}))
    cap.release()
    df = pd.DataFrame(rows)
    df.to_csv(out / 'manifest.csv', index=False)
    return df

SUMMARY = {}
for vpath in sorted(VIDEO_DIR.glob('*.mp4')):
    cands = ALL_CANDIDATES[vpath.name]
    selected, tier = select_with_yield_guarantee(cands, TARGET_K, MIN_OUTPUT)
    df = export_frames(vpath, selected)
    SUMMARY[vpath.name] = df
    n_closeup = int(df['is_closeup'].sum()) if 'is_closeup' in df else 0
    print(f'{vpath.name}: candidates={len(cands)} → selected={len(selected)} '
          f'(closeup={n_closeup}, tier={tier}) → {OUT_DIR / vpath.stem}')

## 9. Visualization: preview selected frames

Vẽ bbox target (xanh lá) và bbox khác (đỏ) lên 6 frame ngẫu nhiên đã chọn để verify.

In [ ]:
import random
random.seed(0)
for vname in SUMMARY:
    vpath = VIDEO_DIR / vname
    df = SUMMARY[vname]
    if df.empty: continue
    picks = df.sample(min(6, len(df))).sort_values('time_s')
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    cap = cv2.VideoCapture(str(vpath))
    for ax, (_, row) in zip(axes.flat, picks.iterrows()):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame_idx))
        ok, f = cap.read()
        if not ok: ax.axis('off'); continue
        sidecar = json.loads((OUT_DIR / vpath.stem / row.filename.replace('.jpg', '.json')).read_text())
        for (x1,y1,x2,y2) in sidecar['target_boxes']:
            cv2.rectangle(f, (x1,y1), (x2,y2), (0,255,0), 3)
        for (x1,y1,x2,y2) in sidecar['other_boxes']:
            cv2.rectangle(f, (x1,y1), (x2,y2), (0,0,255), 2)
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        ax.set_title(f't={row.time_s:.1f}s | sharp={row.sharp_weighted:.0f} | n_tgt={row.n_target}')
        ax.axis('off')
    cap.release()
    fig.suptitle(vname, fontsize=14)
    plt.tight_layout(); plt.show()

## 10. (Optional) Re-tune top-K mà không chạy lại detection

Khi cache đã có, chỉnh `TARGET_K`, các trọng số `W_*`, `MIN_OUTPUT` hoặc `PHASH_HAM_THR` rồi chạy lại cell 8 — tốn vài giây thay vì hàng chục phút.

Để **xóa cache** và chạy lại từ đầu cho 1 video:
```python
(CACHE_DIR / 'M01_white_1080p_candidates.json').unlink(missing_ok=True)
(MASK_DIR  / 'M01_white_1080p_mask.png').unlink(missing_ok=True)
```

## 11. Upload lên Roboflow + auto-annotation workflow

Sau khi notebook xong, mỗi `extracted_frames/<video_name>/` có:
- `*.jpg` — frame chất lượng cao (resolution gốc).
- `*.json` — sidecar với `target_boxes` (cầu thủ target), `max_target_h`, `is_closeup`, scores. Dùng để **lọc** thêm khi annotate (ưu tiên `is_closeup=True` trước).
- `manifest.csv` — overview toàn bộ frame.

### Có cần annotate tất cả 150-250 frame/video không? KHÔNG.

Roboflow có 4 cơ chế auto-annotate. Workflow tôi đề xuất:

| Bước | Công cụ Roboflow | Số frame | Effort |
|---|---|---|---|
| 1. Manual seed | **Smart Polygon** (SAM 2 click-to-mask) | 30-50 close-up frame đầu | ~30-60 phút |
| 2. Train v1 | **Roboflow Train** (YOLOv8/RT-DETR) | dùng 30-50 đó | ~10 phút tự động |
| 3. Auto-label rest | **Label Assist** (v1 model gợi ý bbox) | 100-200 frame còn lại | chỉ verify/correct, ~5s/frame |
| 4. Re-train v2 | **Roboflow Train** | toàn bộ dataset | ~10 phút |
| 5. (option) Active learning | Model deploy → uncertain frame tự gửi về annotate queue | — | — |

### Hoặc dùng foundation model (không cần train trước)

Roboflow **Auto Label** dùng Grounding DINO + SAM 2 → cho text prompt như:
- `"KLG logo on rugby jersey"`
- `"AON sponsor patch on chest"`
- `"flower tomb shipley logo"`
- `"mcp logo on back of jersey"`

Foundation model tự đề xuất bbox + mask cho TẤT CẢ frame, bạn verify. Hoạt động tốt với logo có text rõ; logo trừu tượng cần manual hơn.

### Tips cho việc annotate logo trên áo rugby

- Trong sidecar `*.json`, `target_boxes` chính là vùng có logo. Trên Roboflow API, bạn có thể bulk-import các bbox này làm **annotation hint** (rectangle), sau đó dùng SAM-polygon để refine thành logo polygon chính xác.
- Bỏ qua frame có `is_closeup=False` ở pass đầu — chỉ annotate close-up trước vì độ phân giải logo lớn hơn → model học tốt hơn.
- Sau khi có ~50 frame annotated tốt, train v1 model rồi mới chạy auto-label trên `is_closeup=False` frame để mở rộng dataset.

### Sample rate cho video dài

Notebook đã tự `adaptive_sample_fps`:
- Video highlight < 20 phút: 2 fps
- Match 20-60 phút: 1 fps  
- Match 1-2h: 0.5 fps
- Match 2-3h: 0.33 fps

Có thể override bằng `SAMPLE_FPS_OVERRIDE = 0.5` ở cell 2.